In [ ]:
!pip install implicit==0.7.2

In [1]:
import os
import numpy as np
import scipy.sparse as sp
import torch
import torch.optim as optim
import argparse

from datasets import Dataloader, prepare_interaction_data, split_input_target_interactions
from util import (
    evaluate_ndcg_at_k,
    evaluate_recall_at_k,
    get_checkpoint_name,
    get_results_filepath,
    run_training_loop,
    save_results,
    set_seed,
)


def evaluate_on_split(model, split: sp.csr_matrix, cfg: dict, device: torch.device) -> dict:
    inputs, targets = split_input_target_interactions(split, cfg["target_interaction_ratio"])
    inputs, targets = Dataloader(inputs, cfg["batch_size"], device), Dataloader(targets, cfg["batch_size"], device)
    model.eval()
    recalls = evaluate_recall_at_k(model, inputs, targets, cfg["eval_topk"])
    ndcgs = evaluate_ndcg_at_k(model, inputs, targets, cfg["eval_topk"])
    return {
        "recall": {"mean": float(np.mean(recalls)), "se": float(np.std(recalls) / np.sqrt(len(recalls)))},
        "ndcg": {"mean": float(np.mean(ndcgs)), "se": float(np.std(ndcgs) / np.sqrt(len(ndcgs)))},
    }


def train_elsa(cfg: dict, device: torch.device):
    print(f"Training ELSA model using config {cfg}")

    _, train_csr, val_csr, test_csr, _, _, _, _ = prepare_interaction_data(cfg)
    train_dataloader = Dataloader(train_csr, cfg["batch_size"], device, shuffle=True)
    val_dataloader = Dataloader(val_csr, cfg["batch_size"], device)

    model_class = getattr(importlib.import_module(cfg["model_module"]), cfg["model_class"])
    model = model_class(train_csr.shape[1], cfg["embedding_dim"]).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"], betas=(cfg["beta1"], cfg["beta2"]))

    run_training_loop(model, optimizer, train_dataloader, val_dataloader, cfg, device, save_ckpt=ast.literal_eval(os.environ.get("SAVE_CKPT", "True")))

    results = {"val": evaluate_on_split(model, val_csr, cfg, device), "test": evaluate_on_split(model, test_csr, cfg, device)}
    for split, split_res in results.items():
        for m in split_res.keys():
            print(f"model = {get_checkpoint_name(cfg)} | split = {split} | {m} @ {cfg['eval_topk']} = {split_res[m]['mean']:.6f} +- {split_res[m]['se']:.6f}")
    save_results(results, cfg, get_results_filepath(cfg))



parser = argparse.ArgumentParser(description="Argument parser for ELSA training script.")
parser.add_argument("--dataset", type=str, default="ML-25M", help="Dataset name")
parser.add_argument("--val_user_ratio", type=float, default=0.1, help="Ratio of validation users")
parser.add_argument("--test_user_ratio", type=float, default=0.1, help="Ratio of test users")
parser.add_argument("--target_interaction_ratio", type=float, default=0.2, help="Ratio of interactions used as target")
parser.add_argument("--model_module", type=str, default="elsa", help="Module containing ELSA model")
parser.add_argument("--model_class", type=str, default="ELSA", help="Model class name")
parser.add_argument("--embedding_dim", type=int, default=1024, help="Embedding dimension of ELSA model")
parser.add_argument("--epochs", type=int, default=10, help="Number of epochs")
parser.add_argument("--early_stopping", type=int, default=10, help="Early stopping number of epochs")
parser.add_argument("--batch_size", type=int, default=1024, help="Batch size")
parser.add_argument("--lr", type=float, default=1e-4, help="Learning rate")
parser.add_argument("--beta1", type=float, default=0.9, help="Adam beta_1 coefficient")
parser.add_argument("--beta2", type=float, default=0.99, help="Adam beta_2 coefficient")
parser.add_argument("--eval_topk", type=int, default=20, help="Evalutation top k")
parser.add_argument("--seed", type=float, default=42, help="Random seed")
parser.add_argument("-f", "--fail", type=str, default="", help="bla seed")
cfg = vars(parser.parse_args())
print(cfg)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.mps.is_available() else torch.device("cpu")
set_seed(cfg["seed"])
train_elsa(cfg, device)

{'dataset': 'ML-25M', 'val_user_ratio': 0.1, 'test_user_ratio': 0.1, 'target_interaction_ratio': 0.2, 'model_module': 'elsa', 'model_class': 'ELSA', 'embedding_dim': 1024, 'epochs': 10, 'early_stopping': 10, 'batch_size': 1024, 'lr': 0.0001, 'beta1': 0.9, 'beta2': 0.99, 'eval_topk': 20, 'seed': 42, 'fail': '/home/vojta/.local/share/jupyter/runtime/kernel-e9fcda27-81ec-4167-980c-1d1bb90e6f14.json'}
Training ELSA model using config {'dataset': 'ML-25M', 'val_user_ratio': 0.1, 'test_user_ratio': 0.1, 'target_interaction_ratio': 0.2, 'model_module': 'elsa', 'model_class': 'ELSA', 'embedding_dim': 1024, 'epochs': 10, 'early_stopping': 10, 'batch_size': 1024, 'lr': 0.0001, 'beta1': 0.9, 'beta2': 0.99, 'eval_topk': 20, 'seed': 42, 'fail': '/home/vojta/.local/share/jupyter/runtime/kernel-e9fcda27-81ec-4167-980c-1d1bb90e6f14.json'}
Removing users with < 5 interactions...
Dataset info: users=160776, items=40857, interactions=12448242
Train split info: users=128621, items=40857, interactions=9969

NameError: name 'importlib' is not defined

In [7]:
_, train_csr, val_csr, test_csr, _, _, _, _ = prepare_interaction_data(cfg)

Removing users with < 5 interactions...
Dataset info: users=160776, items=40857, interactions=12448242
Train split info: users=128621, items=40857, interactions=9975428
Val split info: users=16078, items=40857, interactions=1231648
Test split info: users=16077, items=40857, interactions=1241166


In [8]:
train_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 9975428 stored elements and shape (128621, 40857)>

In [9]:
val_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1231648 stored elements and shape (16078, 40857)>

In [10]:
test_csr

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1241166 stored elements and shape (16077, 40857)>

In [11]:
import implicit


In [29]:
class ALSMatrixFactorizer():
    def __init__(self, factors: int, regularization: float, iterations: int, use_gpu: bool, num_threads: int = 0):
        self.model = None
        self.factors = factors
        self.regularization = regularization
        self.iterations = iterations
        self.use_gpu = use_gpu
        #self.item_idx = item_idx
        self.num_threads = num_threads
        self.trained = False

    def name(self):
        return "MF"

    def fit(self, X, training=False):
        self.model = implicit.als.AlternatingLeastSquares(
            factors=self.factors,
            regularization=self.regularization,
            iterations=self.iterations,
            use_gpu=self.use_gpu,
            num_threads=self.num_threads,

        )
        self.model.fit(X)
        self.trained = True

    def get_item_embeddings(self):
        if self.trained:
            return self.model.item_factors

    def get_user_embeddings(self):
        if self.trained:
            return self.model.user_factors
        

In [19]:
als = ALSMatrixFactorizer(factors=1024, regularization=50, iterations=1, use_gpu=False, num_threads=30)

In [20]:
als.fit(train_csr)

  0%|          | 0/20 [00:00<?, ?it/s]

In [34]:
als.get_user_embeddings()

array([[-0.01028632, -0.00217675,  0.00977941, ...,  0.00341783,
         0.01604783, -0.00926033],
       [ 0.04398589,  0.00908446,  0.06595463, ...,  0.0177864 ,
        -0.00012783,  0.01036902],
       [-0.02860086, -0.02170662,  0.02725253, ..., -0.04116611,
         0.00021907,  0.00634884],
       ...,
       [-0.0070745 , -0.01509932,  0.03790789, ...,  0.01428629,
         0.00539028,  0.01187012],
       [-0.00738186,  0.00021155, -0.00521915, ...,  0.01341446,
         0.00505234, -0.00047035],
       [ 0.05423894,  0.00940892, -0.01134132, ...,  0.01729597,
         0.01574939, -0.03206043]], dtype=float32)

In [33]:
als.get_item_embeddings()

array([[ 2.0498051e-01, -2.3240343e-01,  4.6704724e-01, ...,
        -3.3964962e-01,  2.9737669e-01,  5.7292265e-01],
       [ 8.0788232e-02, -1.5418848e-02,  2.7675061e-02, ...,
         9.3922056e-03,  8.6140692e-02,  9.1917194e-02],
       [ 1.0074894e-02, -1.8716330e-02, -6.1898576e-03, ...,
         1.2499245e-02,  1.0867120e-02,  1.2315487e-02],
       ...,
       [-3.4125175e-04, -1.2331759e-04, -1.2720531e-04, ...,
         2.4378448e-04,  3.9441444e-04,  4.7946299e-04],
       [-3.4037273e-04, -1.2243842e-04, -1.2613846e-04, ...,
         2.4393687e-04,  3.9506404e-04,  4.7893645e-04],
       [-3.4044127e-04, -1.2245563e-04, -1.2633031e-04, ...,
         2.4396808e-04,  3.9500627e-04,  4.7969402e-04]], dtype=float32)

In [36]:
recommended_item_ids, scores = als.model.recommend(
            np.arange(val_csr.shape[0]), val_csr,
            recalculate_user=True, filter_already_liked_items=False,
            N=20
        )

In [40]:
recommended_item_ids, scores

(array([[  53,  468,   39, ...,   77,   61,  156],
        [ 557,  149,   54, ..., 1034, 1031,  543],
        [ 151,  540,  533, ...,  730,   45, 1689],
        ...,
        [ 158,  381,   29, ...,  383,  917,  934],
        [ 102,   47,  115, ...,  167,   53,   59],
        [  44,   51,    0, ..., 1692,  167,  549]], dtype=int32),
 array([[0.86743605, 0.86718506, 0.81320506, ..., 0.67740154, 0.6579368 ,
         0.65300274],
        [0.7144715 , 0.7065701 , 0.57834953, ..., 0.15485331, 0.14900476,
         0.14405984],
        [0.61416185, 0.55311954, 0.5351286 , ..., 0.14639418, 0.142519  ,
         0.13958357],
        ...,
        [0.6605614 , 0.6568421 , 0.63236904, ..., 0.29131648, 0.27059892,
         0.21536228],
        [0.8909131 , 0.89060515, 0.8721645 , ..., 0.7581849 , 0.7544646 ,
         0.754056  ],
        [0.8257141 , 0.78655326, 0.7793782 , ..., 0.39688206, 0.367141  ,
         0.35450143]], dtype=float32))

In [43]:
val_user_embeddings = als.model.recalculate_user(np.arange(val_csr.shape[0]), val_csr)

In [44]:
val_user_embeddings.shape

(16078, 1024)